# Tarefa 3.2: Preparação dos Dados e Vetorização

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

dados = {
    'texto': [
        'Compre agora e ganhe um desconto!',
        'Olá, tudo bem? Vamos nos encontrar?',
        'Ganhe dinheiro fácil e rápido, sem esforço.',
        'Convite para um café, me responda por favor.',
        'Promoção imperdível! Clique para ganhar um brinde.',
        'Reunião agendada para amanhã, 10h.'
    ],
    'categoria': ['spam', 'não-spam', 'spam', 'não-spam', 'spam', 'não-spam']
}
df = pd.DataFrame(dados)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['texto'])
y = df['categoria']

print("Dados vetoriais (primeiro documento):")
print(X.toarray()[0])

# Tarefa 3.3: Treinamento do Modelo: Naive Bayes e SVM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

# Instanciando e treinando o modelo Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_treino, y_treino)

# Instanciando e treinando o modelo SVM
modelo_svm = SVC(kernel='linear')
modelo_svm.fit(X_treino, y_treino)

# Tarefa 3.4: Avaliação dos Resultados

In [ ]:
predicoes_nb = modelo_nb.predict(X_teste)
acuracia_nb = accuracy_score(y_teste, predicoes_nb)

predicoes_svm = modelo_svm.predict(X_teste)
acuracia_svm = accuracy_score(y_teste, predicoes_svm)

print(f"Acurácia do modelo Naive Bayes: {acuracia_nb:.2f}")
print(f"Acurácia do modelo SVM: {acuracia_svm:.2f}")

# Exemplo 1 – Dataset pequeno

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

corpus = [
    "o filme é excelente e muito divertido",
    "não gostei do enredo, muito chato",
    "a atuação foi brilhante e o roteiro impecável",
    "péssimo! perdi meu tempo assistindo",
    "a história é ok, mas nada de mais",
    "filme horrível, nem vale a pena",
    "uma obra-prima da sétima arte",
    "o filme tem um final surpreendente e emocionante",
    "este é o pior filme que já vi na vida"
]
labels = ["positivo", "negativo", "positivo", "negativo", "neutro", "negativo", "positivo", "positivo", "negativo"]

X_treino, X_teste, y_treino, y_teste = train_test_split(corpus, labels, test_size=0.3, random_state=42)

vectorizer = TfidfVectorizer(max_features=100)
X_treino_vetorizado = vectorizer.fit_transform(X_treino)
X_teste_vetorizado = vectorizer.transform(X_teste)

# Criando e avaliando o modelo inicial
modelo_base = SVC()
modelo_base.fit(X_treino_vetorizado, y_treino)
previsoes = modelo_base.predict(X_teste_vetorizado)

print("### Análise Inicial do Modelo (linha de base) ###\n")
print(classification_report(y_teste, previsoes, zero_division=0))
print("\nMatriz de Confusão:")
print(confusion_matrix(y_teste, previsoes))

# Otimizando com Validação Cruzada

In [ ]:
scores = cross_val_score(modelo_base, X_treino_vetorizado, y_treino, cv=3, scoring='f1_macro')
print("\n### Avaliação com Validação Cruzada ###\n")
print(f"Scores para cada 'fold': {scores}")
print(f"Média do F1-score com Validação Cruzada: {scores.mean():.2f}")

# A Caça aos Melhores Hiperparâmetros (Grid Search)

In [ ]:
parametros = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid_search = GridSearchCV(SVC(), parametros, cv=3, scoring='f1_macro')
grid_search.fit(X_treino_vetorizado, y_treino)

print("\nBusca Concluída!")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor pontuação (F1-score) encontrada: {grid_search.best_score_:.2f}")

# Exemplo 2 – Dataset maior

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

corpus = [
    "o filme é excelente e muito divertido", "nao gostei do enredo",
    "a atuacao foi brilhante", "pessimo! perdi meu tempo assistindo",
    "a historia e ok, mas nada de mais", "filme horrivel, nem vale a pena",
    "uma obra-prima da setima arte", "o filme tem um final surpreendente e emocionante",
    "este é o pior filme que ja vi na vida", "excelente direcao e fotografia",
    "nao recomendo, um lixo", "filme maravilhoso, amei", "sem graça e previsível",
    "simplesmente incrível, o melhor filme do ano", "que filme ruim, nao entendi a hype",
    "espetacular, recomendo a todos"
] * 20
labels = (["positivo", "negativo", "positivo", "negativo", "neutro", "negativo", "positivo", "positivo", "negativo", "positivo", "negativo", "positivo", "negativo", "positivo", "negativo", "positivo"]) * 20

X_treino, X_teste, y_treino, y_teste = train_test_split(corpus, labels, test_size=0.2, random_state=42, stratify=labels)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=500)
X_treino_vetorizado = vectorizer.fit_transform(X_treino)
X_teste_vetorizado = vectorizer.transform(X_teste)

# Criando e avaliando o modelo
modelo_base = SVC()
modelo_base.fit(X_treino_vetorizado, y_treino)
previsoes = modelo_base.predict(X_teste_vetorizado)

print("### Análise Inicial do Modelo (linha de base) ###\n")
print(classification_report(y_teste, previsoes, zero_division=0))
print("\nMatriz de Confusão:")
print(confusion_matrix(y_teste, previsoes))

# Otimizando com Validação Cruzada

In [ ]:
scores = cross_val_score(modelo_base, X_treino_vetorizado, y_treino, cv=5, scoring='f1_macro')
print("\n### Avaliação com Validação Cruzada ###\n")
print(f"Scores para cada 'fold': {scores.round(2)}")
print(f"Média do F1-score com Validação Cruzada: {scores.mean():.2f}")

# A Caça aos Melhores Hiperparâmetros (Grid Search)

In [ ]:
parametros = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']}
grid_search = GridSearchCV(SVC(), parametros, cv=5, scoring='f1_macro')
grid_search.fit(X_treino_vetorizado, y_treino)

print("\nBusca Concluída!")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor pontuação (F1-score) encontrada: {grid_search.best_score_:.2f}")